# 01 Dataset Validation

Component 2 Phase 1 research notebook.

## 1. Research Dataset Introduction
This notebook validates the balanced Component 2 pilot export before any modelling. The goal is to confirm data shape, relationships, missing values, duplicates, date ranges, reply structure, and video metadata coverage.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not ((PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT / 'shared').exists()):
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'research' / 'component_2' / 'src'
sys.path.insert(0, str(SRC_DIR))

COMMENTS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'component_2' / 'component2_pilot_comments.csv'
VIDEOS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'component_2' / 'component2_pilot_videos.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'component_2'
ANALYSIS_DIR = PROJECT_ROOT / 'data' / 'analysis' / 'component_2' / '01_pilot_2000_comments'
CHART_DIR = ANALYSIS_DIR / 'charts'
for directory in [PROCESSED_DIR, ANALYSIS_DIR, CHART_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

from data_io import (
    COMMENTS_REQUIRED_COLUMNS,
    VIDEOS_REQUIRED_COLUMNS,
    file_sha256,
    load_comments,
    load_videos,
    parse_datetime_columns,
    validate_required_columns,
)
from validation import (
    channel_relationship_mismatches,
    comments_per_channel,
    comments_per_video,
    date_range,
    duplicate_count,
    duplicate_value_summary,
    missing_value_summary,
    reply_summary,
    text_length_stats,
    video_channel_distribution,
    video_match_summary,
)
from eda import save_bar_chart, save_histogram

assert COMMENTS_PATH.exists(), f'Missing comments file: {COMMENTS_PATH}'
assert VIDEOS_PATH.exists(), f'Missing videos file: {VIDEOS_PATH}'
comments = load_comments(COMMENTS_PATH)
videos = load_videos(VIDEOS_PATH)
comments = parse_datetime_columns(comments, ['published_at', 'edited_at'])
videos = parse_datetime_columns(videos, ['published_at'])

## 2. Dataset Source and Provenance
The pilot files are local CSV exports derived from Supabase `research.comments` and `research.videos`, with channel metadata originally from `research.channels`. The sampling method is documented as up to 400 non-empty comments per selected channel, spread across videos, using deterministic `MD5(comment_id)` ordering. Exact extraction details not present in the local CSV are recorded as unknown rather than inferred.

In [2]:
provenance = pd.DataFrame({
    'artifact': ['comments_csv', 'videos_csv'],
    'path': [str(COMMENTS_PATH.relative_to(PROJECT_ROOT)), str(VIDEOS_PATH.relative_to(PROJECT_ROOT))],
    'sha256': [file_sha256(COMMENTS_PATH), file_sha256(VIDEOS_PATH)],
    'source_table': ['research.comments', 'research.videos'],
})
provenance

,artifact,path,sha256,source_table
0,comments_csv,data/raw/component_2/component2_pilot_comments...,481acece343115e4a00163b95df558f3d3a99775b07eab...,research.comments
1,videos_csv,data/raw/component_2/component2_pilot_videos.csv,b11ad9809aecd4500aebfb739ebbfe78c60777e20ae7aa...,research.videos


## 3. Dataset Shape

In [3]:
shape_summary = pd.DataFrame({
    'dataset': ['comments', 'videos'],
    'rows': [len(comments), len(videos)],
    'columns': [comments.shape[1], videos.shape[1]],
})
shape_summary

,dataset,rows,columns
0,comments,2000,9
1,videos,327,8


## 4. Column Names and Data Types

In [4]:
comments_dtypes = comments.dtypes.astype(str).rename('dtype').reset_index().rename(columns={'index': 'column'})
videos_dtypes = videos.dtypes.astype(str).rename('dtype').reset_index().rename(columns={'index': 'column'})
display(comments_dtypes)
display(videos_dtypes)

,column,dtype
0,comment_id,string
1,channel_id,object
2,channel_name,object
3,video_id,object
4,comment_text,object
5,published_at,"datetime64[ns, UTC]"
6,edited_at,"datetime64[ns, UTC]"
7,like_count,int64
8,parent_comment_id,string


,column,dtype
0,video_id,string
1,channel_id,object
2,channel_name,object
3,title,object
4,description,object
5,tags,object
6,published_at,"datetime64[ns, UTC]"
7,duration_seconds,int64


## 5. Required Columns and Missing Values

In [5]:
validate_required_columns(comments, COMMENTS_REQUIRED_COLUMNS, 'comments')
validate_required_columns(videos, VIDEOS_REQUIRED_COLUMNS, 'videos')
comments_missing = missing_value_summary(comments)
videos_missing = missing_value_summary(videos)
display(comments_missing)
display(videos_missing)

,column,missing_count,missing_percent
0,comment_id,0,0.00
1,channel_id,0,0.00
2,channel_name,0,0.00
3,video_id,0,0.00
4,comment_text,0,0.00
5,published_at,0,0.00
6,edited_at,0,0.00
7,like_count,0,0.00
8,parent_comment_id,1633,81.65


,column,missing_count,missing_percent
0,video_id,0,0.0
1,channel_id,0,0.0
2,channel_name,0,0.0
3,title,0,0.0
4,description,0,0.0
5,tags,0,0.0
6,published_at,0,0.0
7,duration_seconds,0,0.0


## 6. Duplicate ID and Text Checks

In [6]:
duplicate_summary = pd.DataFrame({
    'check': ['duplicate_comment_ids', 'duplicate_video_ids', 'duplicate_comment_text_values'],
    'value': [
        duplicate_count(comments, 'comment_id'),
        duplicate_count(videos, 'video_id'),
        int((comments['comment_text'].fillna('').astype(str).value_counts() > 1).sum()),
    ],
})
display(duplicate_summary)
display(duplicate_value_summary(comments, 'comment_text', limit=10))

,check,value
0,duplicate_comment_ids,0
1,duplicate_video_ids,0
2,duplicate_comment_text_values,50


,comment_text,count
0,❤,52
1,❤❤,33
2,❤❤❤,27
3,❤❤❤❤,20
4,❤❤❤❤❤,19
5,❤❤❤❤❤❤,14
6,😂,13
7,❤️,10
8,😂😂😂,9
9,😂😂,6


## 7. Channel Distribution and Commented Video Counts

In [7]:
channel_counts = comments_per_channel(comments)
video_counts = video_channel_distribution(videos)
display(channel_counts)
display(video_counts)
save_bar_chart(channel_counts.set_index('channel_name')['total_comments'], 'Comments per Pilot Channel', 'Comments', CHART_DIR / 'validation_comments_per_channel.png')
save_bar_chart(video_counts.set_index('channel_name')['total_videos'], 'Stored Videos per Pilot Channel', 'Videos', CHART_DIR / 'validation_videos_per_channel.png')

,channel_id,channel_name,total_comments,videos_with_comments
0,UCCK3OZi788Ok44K97WAhLKQ,Ada Derana,400,76
1,UCHKtXGbqRIM0hA4Manb9DNg,Ada Derana Sports,400,38
2,UCOPPH1_whpThgY6e7r3qg-w,FM Derana,400,43
3,UCRDDfbYPHX_GUJ4lcQYTc8A,TV Derana,400,50
4,UCs8gVM6lPhRZk3L_zZmedpw,Dream Star,400,48


,channel_id,channel_name,total_videos
0,UCCK3OZi788Ok44K97WAhLKQ,Ada Derana,100
3,UCRDDfbYPHX_GUJ4lcQYTc8A,TV Derana,76
2,UCOPPH1_whpThgY6e7r3qg-w,FM Derana,51
1,UCHKtXGbqRIM0hA4Manb9DNg,Ada Derana Sports,50
4,UCs8gVM6lPhRZk3L_zZmedpw,Dream Star,50


PosixPath('/Users/nipuninavodyakahandawa/Documents/Framework-for-Underperforming-YouTube-Channels-in-a-Multi-Channel-Media-Organization---J26-DS-334/data/analysis/component_2/charts/validation_videos_per_channel.png')

## 8. Comment-to-Video Matching and Channel Consistency

In [8]:
match_summary = pd.DataFrame([video_match_summary(comments, videos)])
mismatches = channel_relationship_mismatches(comments, videos)
display(match_summary)
print(f'Channel relationship mismatches: {len(mismatches)}')

,commented_videos,matched_videos,missing_video_metadata,unused_stored_videos
0,255,255,0,72


Channel relationship mismatches: 0


## 9. Publication Date Ranges

In [9]:
date_summary = pd.DataFrame([
    {'dataset': 'comments', 'date_column': 'published_at', **date_range(comments, 'published_at')},
    {'dataset': 'comments', 'date_column': 'edited_at', **date_range(comments, 'edited_at')},
    {'dataset': 'videos', 'date_column': 'published_at', **date_range(videos, 'published_at')},
])
date_summary

,dataset,date_column,min,max,missing
0,comments,published_at,2026-01-18 13:16:23+00:00,2026-10-10 07:57:26+00:00,0
1,comments,edited_at,2026-01-18 13:16:23+00:00,2026-10-10 07:57:26+00:00,0
2,videos,published_at,2026-01-18 13:15:07+00:00,2026-10-10 07:54:54+00:00,0


## 10. Reply Versus Top-Level Comments

In [10]:
reply_counts = reply_summary(comments)
reply_counts

,comment_type,count
0,top_level,1633
1,reply,367


## 11. Video Title and Description Availability

In [11]:
video_text_availability = pd.DataFrame({
    'field': ['title', 'description', 'tags'],
    'missing_or_empty': [
        int((videos['title'].isna() | (videos['title'].astype(str).str.strip() == '')).sum()),
        int((videos['description'].isna() | (videos['description'].astype(str).str.strip() == '')).sum()),
        int((videos['tags'].isna() | (videos['tags'].astype(str).str.strip() == '')).sum()),
    ],
})
video_text_availability

,field,missing_or_empty
0,title,0
1,description,0
2,tags,0


## 12. Text Length Statistics

In [12]:
text_lengths = comments['comment_text'].fillna('').astype(str).str.len()
word_counts = comments['comment_text'].fillna('').astype(str).str.split().map(len)
length_stats = pd.DataFrame({
    'char_length': text_lengths.describe(),
    'word_count': word_counts.describe(),
})
display(length_stats)
save_histogram(text_lengths, 'Comment Character Length Distribution', 'Characters', CHART_DIR / 'validation_comment_length_histogram.png')

,char_length,word_count
count,2000.000000,2000.000000
mean,52.889000,8.446000
std,141.654738,21.812725
min,1.000000,1.000000
25%,9.000000,1.000000
50%,26.000000,4.000000
75%,55.000000,9.000000
max,3238.000000,486.000000


PosixPath('/Users/nipuninavodyakahandawa/Documents/Framework-for-Underperforming-YouTube-Channels-in-a-Multi-Channel-Media-Organization---J26-DS-334/data/analysis/component_2/charts/validation_comment_length_histogram.png')

## 13. Data Quality Summary and Outputs

In [13]:
quality_summary = pd.DataFrame({
    'metric': [
        'comments_rows', 'videos_rows', 'distinct_comment_ids', 'distinct_video_ids_in_comments',
        'stored_video_ids', 'missing_video_metadata', 'channel_mismatches', 'empty_comment_texts',
        'duplicate_comment_ids', 'duplicate_video_ids'
    ],
    'value': [
        len(comments), len(videos), comments['comment_id'].nunique(), comments['video_id'].nunique(),
        videos['video_id'].nunique(), match_summary.loc[0, 'missing_video_metadata'], len(mismatches),
        int((comments['comment_text'].isna() | (comments['comment_text'].astype(str).str.strip() == '')).sum()),
        duplicate_count(comments, 'comment_id'), duplicate_count(videos, 'video_id'),
    ],
})
quality_summary.to_csv(ANALYSIS_DIR / 'dataset_quality_summary.csv', index=False)
comments.to_csv(PROCESSED_DIR / 'validated_comments.csv', index=False)
videos.to_csv(PROCESSED_DIR / 'validated_videos.csv', index=False)
provenance.to_csv(ANALYSIS_DIR / 'dataset_provenance_hashes.csv', index=False)
quality_summary

,metric,value
0,comments_rows,2000
1,videos_rows,327
2,distinct_comment_ids,2000
3,distinct_video_ids_in_comments,255
4,stored_video_ids,327
5,missing_video_metadata,0
6,channel_mismatches,0
7,empty_comment_texts,0
8,duplicate_comment_ids,0
9,duplicate_video_ids,0


## 14. Limitations and Observations
This validation confirms structural quality and relationships, but it does not prove audience-demand usefulness. The pilot is deterministic and balanced by selected channels; it should not be described as a random representative sample of all viewers or all videos. Manual review and labelling are still required before demand-intent modelling.